# Information Retrieval — Phase 2
**Team:** Yanis Hemdane, Rayane Khatim, Nour el imene Khelassi, Aya Chihoub

**What this notebook does:**
1. Trains a classifier on documents **and** train queries (fixes the domain gap)
2. Retrieves with BM25+ and `multi-qa-mpnet-base-dot-v1` embeddings
3. Fuses both with Reciprocal Rank Fusion (RRF)
4. Filters results by predicted query category
5. Auto-tunes k on the training set
6. Writes `submission.csv`

**How to run:** click *Run All* (or Shift+Enter through each cell). The last cell saves `submission.csv` in `/kaggle/working/`.

---
## Cell 1 — Install dependencies

In [ ]:
!pip install -q rank_bm25 sentence-transformers tqdm

---
## Cell 2 — Imports

In [ ]:
import csv
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Plus
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm

print('All imports OK')

---
## Cell 3 — Configuration
You can tweak these values before running.

In [ ]:
# ── How many candidates to fetch before filtering ──────────────────────────
POOL = 50          # retrieve top-50, then filter down to K

# ── Final number of docs returned per query ────────────────────────────────
# Set TUNE_K = True to let the notebook find the best K automatically.
# If False, K below is used directly.
TUNE_K = True
K = 10             # starting value / fallback if TUNE_K is False

# ── Embedding model ────────────────────────────────────────────────────────
# multi-qa-mpnet-base-dot-v1 is trained specifically for retrieval tasks.
# Falls back to all-MiniLM-L6-v2 automatically if download fails.
EMB_MODEL = 'multi-qa-mpnet-base-dot-v1'

# ── Cache directory for embeddings (avoids re-encoding on re-run) ──────────
CACHE_DIR = Path('/kaggle/working/cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Output path ────────────────────────────────────────────────────────────
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

print(f'POOL={POOL}  K={K}  TUNE_K={TUNE_K}')
print(f'Embedding model: {EMB_MODEL}')
print(f'Output: {OUTPUT_PATH}')

---
## Cell 4 — Find and load data

In [ ]:
def find_data_dir() -> Path:
    """Locate the competition data folder regardless of how Kaggle mounts it."""
    candidates = [
        Path('/kaggle/input/retrieval-engine-competition'),
        Path('/kaggle/input'),
        Path('data/retrieval-engine-competition'),
        Path('data'),
        Path('.'),
    ]
    for c in candidates:
        if (c / 'docs.json').is_file():
            return c
    for root, _, files in os.walk('/kaggle/input'):
        if 'docs.json' in files:
            return Path(root)
    raise FileNotFoundError('docs.json not found — add the competition dataset under Add Data.')

DATA_DIR = find_data_dir()
print(f'Data directory: {DATA_DIR}')

df_docs          = pd.read_json(DATA_DIR / 'docs.json')
df_queries_train = pd.read_json(DATA_DIR / 'queries_train.json')
df_queries_test  = pd.read_json(DATA_DIR / 'queries_test.json')

with open(DATA_DIR / 'qgts_train.json', 'r') as f:
    raw_gt = json.load(f)
ground_truth = {
    qid: [item['doc_id'] for item in data['relevant_doc_ids']]
    for qid, data in raw_gt.items()
}

print(f'Documents       : {len(df_docs):,}')
print(f'Train queries   : {len(df_queries_train)}')
print(f'Test queries    : {len(df_queries_test)}')
print(f'Ground truth    : {len(ground_truth)} queries')

---
## Cell 5 — Text preprocessing

In [ ]:
def merge_fields(row) -> str:
    """Combine title, text, and tags into a single content string."""
    title    = str(row.get('title', '') or '')
    text     = str(row.get('text',  '') or '')
    tags     = row.get('tags', [])
    tags_str = ' '.join(tags) if isinstance(tags, list) else str(tags or '')
    return ' '.join(f'{title} {text} {tags_str}'.split())

def clean_text(text: str) -> str:
    """Lowercase and strip punctuation (used for BM25 tokenisation)."""
    return re.sub(r'[^\w\s]', '', text.lower())

for df in (df_docs, df_queries_train, df_queries_test):
    df['content']       = df.apply(merge_fields, axis=1)
    df['content_clean'] = df['content'].apply(clean_text)

doc_ids         = df_docs['id'].tolist()
query_ids_train = df_queries_train['id'].tolist()
query_ids_test  = df_queries_test['id'].tolist()
id_to_cat       = dict(zip(df_docs['id'], df_docs['category']))

print('Preprocessing done.')
print(f'Example doc content (first 120 chars): {df_docs["content"].iloc[0][:120]}')

---
## Cell 6 — Train the category classifier

**Key improvement over Phase 1:** we train on both documents *and* training queries.
Test queries are short questions — very different in style from documents.
Including training queries closes that gap and pushes accuracy from ~0.65 to ~0.80+.

In [ ]:
from sklearn.metrics import classification_report

# Combine docs + train queries for training
all_contents = df_docs['content'].tolist() + df_queries_train['content'].tolist()
all_labels   = df_docs['category'].tolist() + df_queries_train['category'].tolist()

print(f'Training on {len(all_contents):,} samples ({len(df_docs):,} docs + {len(df_queries_train)} queries)')
print(f'Categories: {sorted(set(all_labels))}')

# TF-IDF with bigrams captures short phrases like 'stack overflow', 'game engine'
clf_vec = TfidfVectorizer(max_features=15000, sublinear_tf=True, ngram_range=(1, 2))
X_train = clf_vec.fit_transform(all_contents)

clf = LogisticRegression(
    max_iter=1000, random_state=42, solver='lbfgs',
    class_weight='balanced', C=5.0
)
clf.fit(X_train, all_labels)

# Evaluate on train queries only (the ones we have labels for)
X_eval = clf_vec.transform(df_queries_train['content'].tolist())
pred_train_eval = clf.predict(X_eval)
true_train_eval = df_queries_train['category'].tolist()

accuracy = (pred_train_eval == true_train_eval).mean()
print(f'\nTrain-query accuracy: {accuracy:.4f}')
print('\nClassification report:')
print(classification_report(true_train_eval, pred_train_eval))

# Predict categories for all test queries (needed for submission)
X_test_clf = clf_vec.transform(df_queries_test['content'].tolist())
pred_cats_test  = clf.predict(X_test_clf).tolist()
pred_cats_train = clf.predict(clf_vec.transform(df_queries_train['content'].tolist())).tolist()

---
## Cell 7 — BM25+ retrieval

In [ ]:
print(f'Building BM25+ index over {len(df_docs):,} documents…')
tokenized_corpus = [doc.split() for doc in df_docs['content_clean']]
bm25 = BM25Plus(tokenized_corpus)
print('BM25+ index ready.')

def retrieve_bm25(queries_clean: list, pool: int) -> list:
    results = []
    for q in tqdm(queries_clean, desc='BM25+ retrieval'):
        scores  = bm25.get_scores(q.split())
        top_idx = np.argsort(scores)[-pool:][::-1]
        results.append([doc_ids[j] for j in top_idx])
    return results

bm25_train = retrieve_bm25(df_queries_train['content_clean'].tolist(), POOL)
bm25_test  = retrieve_bm25(df_queries_test['content_clean'].tolist(),  POOL)
print(f'BM25+ done. Pool size per query: {POOL}')

---
## Cell 8 — Embedding retrieval

**Key improvement:** `multi-qa-mpnet-base-dot-v1` is trained specifically
on question-to-passage retrieval (MS MARCO, Natural Questions, etc.).
It understands intent much better than the general-purpose MiniLM used in Phase 1.

In [ ]:
print(f'Loading embedding model: {EMB_MODEL}')
try:
    model = SentenceTransformer(EMB_MODEL)
    print(f'Model loaded. Embedding dim: {model.get_sentence_embedding_dimension()}')
except Exception as e:
    print(f'Could not load {EMB_MODEL}: {e}')
    print('Falling back to all-MiniLM-L6-v2')
    EMB_MODEL = 'all-MiniLM-L6-v2'
    model = SentenceTransformer(EMB_MODEL)

model_tag = EMB_MODEL.replace('/', '_').replace('-', '_')

def get_or_encode(texts: list, cache_path: Path, label: str) -> np.ndarray:
    """Load cached embeddings or encode fresh. Saves to cache for re-runs."""
    if cache_path.is_file():
        print(f'  Loading cached {label} from {cache_path.name}')
        return np.load(str(cache_path))
    print(f'  Encoding {label} ({len(texts):,} items)…')
    embs = model.encode(texts, show_progress_bar=True, batch_size=64, normalize_embeddings=True)
    np.save(str(cache_path), embs)
    print(f'  Saved to {cache_path.name}')
    return embs

doc_embs     = get_or_encode(df_docs['content'].tolist(),          CACHE_DIR / f'doc_{model_tag}.npy',      'documents')
train_q_embs = get_or_encode(df_queries_train['content'].tolist(), CACHE_DIR / f'train_q_{model_tag}.npy',  'train queries')
test_q_embs  = get_or_encode(df_queries_test['content'].tolist(),  CACHE_DIR / f'test_q_{model_tag}.npy',   'test queries')

print(f'\nDoc embeddings shape : {doc_embs.shape}')
print(f'Train Q shape        : {train_q_embs.shape}')
print(f'Test Q shape         : {test_q_embs.shape}')

In [ ]:
def retrieve_embeddings(q_embs: np.ndarray, pool: int) -> list:
    """
    Cosine similarity retrieval (both arrays already L2-normalised,
    so dot product == cosine similarity).
    """
    results = []
    for i in tqdm(range(len(q_embs)), desc='Embedding retrieval'):
        sims    = q_embs[i] @ doc_embs.T
        top_idx = np.argsort(sims)[-pool:][::-1]
        results.append([doc_ids[j] for j in top_idx])
    return results

emb_train = retrieve_embeddings(train_q_embs, POOL)
emb_test  = retrieve_embeddings(test_q_embs,  POOL)
print('Embedding retrieval done.')

---
## Cell 9 — Reciprocal Rank Fusion (RRF)

RRF merges BM25+ and embedding results without needing to tune
score scales (BM25 and cosine similarity are in completely different
units). Each document gets `1 / (60 + rank)` from each method; we
sort by total score.

In [ ]:
def reciprocal_rank_fusion(ranked_lists: list, k: int = 60) -> list:
    """Merge ranked doc-id lists using Reciprocal Rank Fusion."""
    scores = {}
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked, 1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=lambda x: scores[x], reverse=True)

print('Fusing BM25+ and embedding results with RRF…')
fused_train = [
    reciprocal_rank_fusion([bm25_train[i], emb_train[i]])
    for i in tqdm(range(len(query_ids_train)), desc='RRF train')
]
fused_test = [
    reciprocal_rank_fusion([bm25_test[i], emb_test[i]])
    for i in tqdm(range(len(query_ids_test)), desc='RRF test')
]
print(f'Fusion done. Fused pool size per query: up to {POOL * 2}')

---
## Cell 10 — Category filtering

**Key improvement over Phase 1 reranking:** instead of just reordering
the existing top-k, we now *filter* the larger pool to keep only
same-category documents before trimming to k.

If too few same-category docs survive (< k/2), we fall back to the
unfiltered pool so we never lose coverage.

In [ ]:
def category_filter(pool_ids: list, pred_cat: str, k: int, fallback: list) -> list:
    """
    Keep docs whose category == pred_cat, then take top-k.
    Falls back to unfiltered pool if fewer than k//2 docs survive.
    """
    same_cat = [d for d in pool_ids if id_to_cat.get(d) == pred_cat]
    if len(same_cat) >= max(k // 2, 3):
        return same_cat[:k]
    return fallback[:k]  # soft fallback

# We'll apply filtering after k is decided (see next cell)

---
## Cell 11 — Auto-tune k

Tests k ∈ {5, 10, 15, 20, 25, 30, 40, 50} on the training queries
and picks the k that maximises the Kaggle score formula.

The Kaggle score = 0.25 × (Recall + Precision + MRR + Accuracy).
Recall grows with k, but Precision shrinks — the sweet spot varies
by dataset.

In [ ]:
def compute_metrics(topk_lists: list, query_ids: list) -> dict:
    """Compute Recall, Precision, MRR, Accuracy and Kaggle score."""
    R, P, M, A = [], [], [], []
    for i, qid in enumerate(query_ids):
        rel = set(ground_truth.get(qid, []))
        ret = topk_lists[i]
        ret_set = set(ret)
        hit = ret_set & rel

        R.append(len(hit) / len(rel)        if rel else 0.0)
        P.append(len(hit) / len(ret)        if ret else 0.0)
        A.append(1.0 if hit else 0.0)

        mrr_val = 0.0
        for rank, d in enumerate(ret, 1):
            if d in rel:
                mrr_val = 1.0 / rank
                break
        M.append(mrr_val)

    return {
        'Recall':    float(np.mean(R)),
        'Precision': float(np.mean(P)),
        'MRR':       float(np.mean(M)),
        'Accuracy':  float(np.mean(A)),
        'Kaggle':    float(np.mean([0.25*(r+p+m+a) for r,p,m,a in zip(R,P,M,A)])),
    }

if TUNE_K:
    print('Auto-tuning k on training queries…')
    print(f'{"k":>4}  {"Kaggle":>8}  {"Recall":>8}  {"Precision":>10}  {"MRR":>8}')
    print('-' * 50)
    best_k, best_score = K, 0.0
    for k_try in [5, 10, 15, 20, 25, 30, 40, 50]:
        # apply filter then trim
        trimmed = [
            category_filter(fused_train[i], pred_cats_train[i], k_try, fused_train[i])
            for i in range(len(query_ids_train))
        ]
        m = compute_metrics(trimmed, query_ids_train)
        print(f'{k_try:>4}  {m["Kaggle"]:>8.4f}  {m["Recall"]:>8.4f}  {m["Precision"]:>10.4f}  {m["MRR"]:>8.4f}')
        if m['Kaggle'] > best_score:
            best_score, best_k = m['Kaggle'], k_try
    K = best_k
    print(f'\n→ Best k = {K}  (estimated Kaggle score: {best_score:.4f})')
else:
    print(f'Using fixed k = {K}  (set TUNE_K = True to auto-select)')

---
## Cell 12 — Final evaluation on training set

In [ ]:
# Build the final train lists with the chosen K
final_train = [
    category_filter(fused_train[i], pred_cats_train[i], K, fused_train[i])
    for i in range(len(query_ids_train))
]

m = compute_metrics(final_train, query_ids_train)

# Classifier accuracy on training queries (this is what Kaggle sees)
clf_acc = float((np.array(pred_cats_train) == np.array(df_queries_train['category'].tolist())).mean())

# Estimated Kaggle score (includes classifier accuracy component)
estimated = 0.25 * (m['Recall'] + m['Precision'] + m['MRR'] + clf_acc)

print('=' * 55)
print(f'FINAL EVALUATION  (k={K})')
print('=' * 55)
print(f'Recall                   : {m["Recall"]:.4f}')
print(f'Precision                : {m["Precision"]:.4f}')
print(f'MRR                      : {m["MRR"]:.4f}')
print(f'Retrieval Accuracy       : {m["Accuracy"]:.4f}')
print(f'Classifier Accuracy      : {clf_acc:.4f}')
print('-' * 55)
print(f'Estimated Kaggle score   : {estimated:.4f}')
print('=' * 55)

---
## Cell 13 — Build test predictions and write submission.csv

In [ ]:
# Apply the same pipeline to test queries
final_test = [
    category_filter(fused_test[i], pred_cats_test[i], K, fused_test[i])
    for i in range(len(query_ids_test))
]

# Write submission.csv
with open(OUTPUT_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['query_id', 'relevant_doc_ids', 'category'])
    for qid, doc_list, cat in zip(query_ids_test, final_test, pred_cats_test):
        writer.writerow([qid, json.dumps(doc_list), cat])

# Verify the file
n_rows  = len(query_ids_test)
n_bytes = OUTPUT_PATH.stat().st_size

print(f'submission.csv written to: {OUTPUT_PATH}')
print(f'Rows  : {n_rows}')
print(f'Size  : {n_bytes:,} bytes')
print()
print('First 3 rows:')
with open(OUTPUT_PATH, 'r') as f:
    for i, line in enumerate(f):
        if i >= 4: break
        print(' ', line.strip()[:120])